# 1. Configuration

## 1.1 Imports

In [1]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

## 1.2 Project Paths

In [2]:
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "utils" / "paths.py").exists())
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from utils.paths import HUMAN_REFERENCE_DIR, PREPROCESSED_DIR, RESULTS_DIR as ALL_RESULTS_DIR
from utils.screen_display import show_screen

RESULTS_DIR = ALL_RESULTS_DIR / "critiques"
RESPONSES_DIR = RESULTS_DIR / "responses"
MATCHING_ROOT_DIR = RESULTS_DIR / "matching"

## 1.3 Select LLM Run

In [ ]:
LLM_NAME = "gpt5"  # "gpt5" | "claude4_7" | "gemini3_1pro"
RUN_LABEL = "zero_shot-all_tasks"

HUMAN_REFERENCE_FILE = HUMAN_REFERENCE_DIR / "human_comments_semantic_dedup.parquet"
LLM_INPUT_PATH = RESPONSES_DIR / f"critiques-{RUN_LABEL}-{LLM_NAME}-responses.parquet"

print(f"LLM: {LLM_NAME}")
print(f"LLM input path: {LLM_INPUT_PATH}")

## 1.4 Output Paths

In [ ]:
MATCHING_OUTPUT_DIR = MATCHING_ROOT_DIR / LLM_NAME / RUN_LABEL
MATCHING_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_STEM = f"{RUN_LABEL}-{LLM_NAME}"
ALL_PAIRS_PATH = MATCHING_OUTPUT_DIR / f"all_comment_pairs-{OUTPUT_STEM}.parquet"

print(f"Matching output folder: {MATCHING_OUTPUT_DIR}")

# 2. Load Data And Build Comment Pairs

In [5]:
from utils.comments.pairs import build_comment_pairs, expand_human_comments, expand_llm_comments

## 2.1 Load And Expand Comments

In [ ]:
if ALL_PAIRS_PATH.exists():
    pairs_df = pd.read_parquet(ALL_PAIRS_PATH)
    print(f"Loaded all pairs: {ALL_PAIRS_PATH}")
else:
    pairs_df = build_comment_pairs(
        expand_human_comments(pd.read_parquet(HUMAN_REFERENCE_FILE)),
        expand_llm_comments(pd.read_parquet(LLM_INPUT_PATH), LLM_NAME),
        LLM_NAME,
    )
    pairs_df.to_parquet(ALL_PAIRS_PATH, index=False)
    print(f"Saved all pairs: {ALL_PAIRS_PATH}")

print(f"Screen tasks with expert and {LLM_NAME} comments: {pairs_df['screen_task_id'].nunique():,}")
print(f"Expert comments: {pairs_df['expert_comment_id'].nunique():,}")
print(f"LLM comments: {pairs_df['llm_comment_id'].nunique():,}")
print(f"Within-task expert-LLM pairs: {len(pairs_df):,}")
pairs_df.head(1)

# 3. Descriptive Analysis

## 3.1 Comment And Pair Counts Per Task

In [9]:
task_volume_df = (
    pairs_df.groupby(["app_category", "screen_id", "screen_task_id"], dropna=False)
    .agg(
        expert_comments=("expert_comment_id", "nunique"),
        llm_comments=("llm_comment_id", "nunique"),
        pair_count=("pair_id", "size"),
    )
    .reset_index()
)

display(task_volume_df[["expert_comments", "llm_comments", "pair_count"]].describe().round(3))

,expert_comments,llm_comments,pair_count
count,2952.000,2952.000,2952.000
mean,3.555,3.381,12.200
std,1.742,0.854,7.156
min,1.000,1.000,1.000
25%,2.000,3.000,6.000
50%,3.000,3.000,12.000
75%,5.000,4.000,16.000
max,11.000,6.000,50.000


## 3.2 App Category Statistics

In [10]:
app_category_summary_df = task_volume_df.groupby("app_category", dropna=False).agg(
    screens=("screen_id", "nunique"),
    screen_tasks=("screen_task_id", "nunique"),
    expert_comments=("expert_comments", "sum"),
    llm_comments=("llm_comments", "sum"),
    pair_count=("pair_count", "sum"),
)
per_task_df = (
    app_category_summary_df[["expert_comments", "llm_comments", "pair_count"]]
    .div(app_category_summary_df["screen_tasks"], axis=0)
    .round(1)
    .add_suffix("_per_task")
)

display(app_category_summary_df.join(per_task_df).sort_values(["screen_tasks", "pair_count"], ascending=False))

,screens,screen_tasks,expert_comments,llm_comments,pair_count,expert_comments_per_task,llm_comments_per_task,pair_count_per_task
app_category,,,,,,,,
Health & Fitness,80,241,858,848,3035,3.6,3.5,12.6
Shopping,78,229,782,761,2636,3.4,3.3,11.5
Travel & Local,68,198,681,666,2323,3.4,3.4,11.7
Social,66,191,702,624,2349,3.7,3.3,12.3
Sports,64,190,723,642,2463,3.8,3.4,13.0
Education,61,180,651,608,2243,3.6,3.4,12.5
Communication,50,147,514,489,1731,3.5,3.3,11.8
Entertainment,48,143,493,479,1698,3.4,3.3,11.9
Weather,46,135,488,473,1737,3.6,3.5,12.9


# 4. LLMJudge Pair Matching

## 4.1 Configure LLMJudge Runs

In [ ]:
from utils.batch_manager import BatchConfig, BatchManager, build_batch_adapter
from utils.comments.judge import (
    DIAGNOSTIC_COLUMNS,
    MATCH_REVIEW,
    compare_judges,
    failed_ids,
    finalize_diagnostics,
    finalize_match_judges,
    iter_requests,
    read_results,
)
from utils.models_setup import resolve_model_config, setup_client

LLMJUDGE_IMAGE_FORMAT = "jpeg"
LLMJUDGE_MAX_TOKENS = 2048
LLMJUDGE_TEMPERATURE = 0.0
LLMJUDGE_REVIEW_STEM = f"human_llm_match_{OUTPUT_STEM}"

llmjudge_dir = MATCHING_OUTPUT_DIR / "llmjudge_matching"
LLMJUDGE_MATCHING_PAIRS_PATH = llmjudge_dir / f"llmjudge_matching_pairs-{OUTPUT_STEM}.parquet"
LLMJUDGE_MANUAL_OVERRIDES_PATH = llmjudge_dir / f"llmjudge_manual_relation_overrides-{OUTPUT_STEM}.csv"

# Batch provider -> model config; each judge's files live under llmjudge_dir/<model_short>.
llmjudge_configs = {provider: resolve_model_config(provider) for provider in ("openai", "gemini")}


def llmjudge_manager(provider: str, *, with_client: bool = False) -> BatchManager:
    """Batch manager for one judge; the API client is only needed to submit or check batches."""
    config = llmjudge_configs[provider]
    settings = dict(temperature=LLMJUDGE_TEMPERATURE, max_tokens=LLMJUDGE_MAX_TOKENS, system_message=MATCH_REVIEW.system_prompt)
    client = None
    if with_client:
        client, _ = setup_client(provider, model=config["model_version"], model_short=config["model_short"], **settings)
    return BatchManager(build_batch_adapter(client, BatchConfig(
        provider=provider,
        model=config["model_version"],
        batch_main_dir=llmjudge_dir / config["model_short"],
        selected_tasks=LLMJUDGE_REVIEW_STEM,
        image_format=LLMJUDGE_IMAGE_FORMAT,
        **settings,
    )))


def load_manual_relations() -> dict[str, str]:
    """Manually reviewed relations by pair_id; every review is saved here immediately."""
    if not LLMJUDGE_MANUAL_OVERRIDES_PATH.exists():
        return {}
    return pd.read_csv(LLMJUDGE_MANUAL_OVERRIDES_PATH).set_index("pair_id")["manual_relation"].to_dict()


for provider, config in llmjudge_configs.items():
    print(f"{config['model_short']}: {config['model_version']} -> {llmjudge_dir / config['model_short']}")

## 4.2 Prepare Candidate Pairs

In [11]:
pairs_df["custom_id"] = pairs_df["expert_comment_id"] + "::" + pairs_df["llm_comment_id"]
print(f"Pairs to judge: {len(pairs_df):,}")

Pairs to judge: 64,266


## 4.3 Build LLMJudge Requests

Skipped for a judge whose request files already exist; delete them to rebuild.

In [ ]:
base64_screen_lookup = None
for provider, config in llmjudge_configs.items():
    manager = llmjudge_manager(provider)
    if any(manager.requests_dir.glob(f"requests-{LLMJUDGE_REVIEW_STEM}-batch_*.jsonl")):
        print(f"{config['model_short']}: request files already exist in {manager.requests_dir}")
        continue
    if base64_screen_lookup is None:
        base64_screen_lookup = (
            pd.read_parquet(PREPROCESSED_DIR / "screenshots.parquet", columns=["screen_id", "base64_screen"])
            .set_index("screen_id")["base64_screen"].to_dict()
        )
        if missing_screen_ids := set(pairs_df["screen_id"]) - set(base64_screen_lookup):
            raise ValueError(f"Missing base64 screenshots for {len(missing_screen_ids)} screen_id values.")
    batch_manifest = manager.write_request_batches_jsonl(
        iter_requests(
            pairs_df,
            MATCH_REVIEW,
            screens=base64_screen_lookup,
            provider=provider,
            model=config["model_version"],
            image_format=LLMJUDGE_IMAGE_FORMAT,
            temperature=LLMJUDGE_TEMPERATURE,
            max_tokens=LLMJUDGE_MAX_TOKENS,
        ),
        max_mb_per_file=180,
        output_stem=LLMJUDGE_REVIEW_STEM,
    )
    request_count = sum(row["requests"] for row in batch_manifest)
    if request_count != len(pairs_df):
        raise ValueError(f"{config['model_short']}: wrote {request_count:,} requests for {len(pairs_df):,} pairs.")

## 4.4 Preview LLMJudge Prompt

In [ ]:
llmjudge_manager("openai").preview_request(index=0, display=True)

## 4.5 Submit And Retrieve LLMJudge Batches

In [6]:
def run_llmjudge_batches(provider: str, *, parallel: bool = True, start_from_batch: int = 1, sleep_minutes: int = 2):
    manager = llmjudge_manager(provider, with_client=True)
    run = manager.process_batches_in_parallel if parallel else manager.process_batches_sequentially
    return run(start_from_batch=start_from_batch, sleep_minutes=sleep_minutes)

### 4.5a OpenAI Judge

In [ ]:
run_llmjudge_batches("openai")

### 4.5b Gemini Judge

In [ ]:
run_llmjudge_batches("gemini")

## 4.6 Load And Compare LLMJudge Responses

In [14]:
llmjudge_results = {
    config["model_short"]: read_results(
        sorted(llmjudge_manager(provider).responses_dir.glob(f"responses-{LLMJUDGE_REVIEW_STEM}-batch_*.jsonl")), MATCH_REVIEW
    )
    for provider, config in llmjudge_configs.items()
}

for results_df in llmjudge_results.values():
    results_df["custom_id"] = results_df["custom_id"].str.split("::").str[-2:].str.join("::")
llmjudge_failed_ids = {judge: failed_ids(results_df, pairs_df["custom_id"]) for judge, results_df in llmjudge_results.items()}
for judge, results_df in llmjudge_results.items():
    print(f"{judge}: {len(results_df):,} responses, {len(llmjudge_failed_ids[judge]):,} pairs without a valid answer")
if any(llmjudge_failed_ids.values()):
    raise ValueError("Some pairs lack a valid judge answer; inspect them in section 4.6.1.")

left_judge, right_judge = llmjudge_results
left_relation_col, right_relation_col = f"relation__{left_judge}", f"relation__{right_judge}"
llmjudge_comparison_df = compare_judges(llmjudge_results, ["relation", "confidence", *DIAGNOSTIC_COLUMNS]).merge(
    pairs_df[[
        "custom_id",
        "pair_id",
        "screen_task_id",
        "screen_id",
        "task",
        "expert_comment_id",
        "llm_comment_id",
        "expert_observed_issue",
        "llm_observed_issue",
        "expert_bounding_box",
    ]],
    on="custom_id",
    validate="one_to_one",
)
same_relation = llmjudge_comparison_df[left_relation_col].eq(llmjudge_comparison_df[right_relation_col])

print("Compared judges:", left_judge, "vs", right_judge)
print("Same relation rate:", same_relation.mean())
print("Disagreement rows:", int((~same_relation).sum()))
display(pd.crosstab(llmjudge_comparison_df[left_relation_col], llmjudge_comparison_df[right_relation_col], dropna=False))

gpt5_2: 64,266 responses, 0 pairs without a valid answer
gemini_3_7_flash: 64,266 responses, 0 pairs without a valid answer
Compared judges: gpt5_2 vs gemini_3_7_flash
Same relation rate: 0.749323125758566
Disagreement rows: 16110


relation__gemini_3_7_flash,A_broader,B_broader,contradiction,equivalent,no_match,partial_overlap,related_but_different
relation__gpt5_2,,,,,,,
A_broader,122,76,1,143,189,73,843
B_broader,22,779,0,197,172,42,538
contradiction,0,1,36,0,35,0,22
equivalent,7,76,0,425,5,5,31
no_match,0,26,8,4,37467,3,6989
partial_overlap,9,111,1,86,388,124,1698
related_but_different,2,123,31,51,4050,52,9203


### 4.6.1 Inspect Invalid Answers

In [15]:
for judge, results_df in llmjudge_results.items():
    invalid_df = results_df[results_df["error"].notna()]
    missing_count = len(llmjudge_failed_ids[judge]) - len(invalid_df)
    print(f"{judge}: {len(invalid_df):,} invalid answers, {missing_count:,} pairs without a response")
    if len(invalid_df):
        print(invalid_df["error"].value_counts().to_string())
        display(invalid_df.head(10))

gpt5_2: 0 invalid answers, 0 pairs without a response
gemini_3_7_flash: 0 invalid answers, 0 pairs without a response


### 4.6.2 Summarize LLMJudge Disagreements

In [16]:
def relation_pair_label(df: pd.DataFrame) -> pd.Series:
    """Order-independent label of the two judges' relations, e.g. "equivalent | no_match"."""
    relations = df[[left_relation_col, right_relation_col]].astype(str)
    return relations.min(axis=1) + " | " + relations.max(axis=1)


display(relation_pair_label(llmjudge_comparison_df[~same_relation]).value_counts().rename_axis("relation_pair").reset_index(name="n"))

,relation_pair,n
0,no_match | related_but_different,11039
1,partial_overlap | related_but_different,1750
2,A_broader | related_but_different,845
3,B_broader | related_but_different,661
4,no_match | partial_overlap,391
5,B_broader | equivalent,273
6,B_broader | no_match,198
7,A_broader | no_match,189
8,B_broader | partial_overlap,153
9,A_broader | equivalent,150


## 4.7 Interactive Review Of Judge Disagreements

Every decision is saved to the manual overrides CSV right away.

In [17]:
SELECTED_MANUAL_RELATION_PAIR = "A_broader | contradiction"  # a relation_pair from 4.6.2, or None for all
MANUAL_REVIEW_BATCH_SIZE = 50
RELATION_SHORTCUTS = {
    "e": "equivalent",
    "a": "A_broader",
    "b": "B_broader",
    "p": "partial_overlap",
    "r": "related_but_different",
    "n": "no_match",
    "c": "contradiction",
}

manual_relations = load_manual_relations()
remaining_df = finalize_match_judges(llmjudge_comparison_df, left_judge, right_judge, manual_relations)
remaining_df = remaining_df[remaining_df["decision_source"].eq("manual_review")]
queue_df = remaining_df
if SELECTED_MANUAL_RELATION_PAIR is not None:
    queue_df = queue_df[relation_pair_label(queue_df).eq(SELECTED_MANUAL_RELATION_PAIR)]
queue_df = queue_df.head(MANUAL_REVIEW_BATCH_SIZE)

print(f"Manual relations saved so far: {len(manual_relations):,}")
print(f"Unresolved disagreements: {len(remaining_df):,}; queued now: {len(queue_df):,}")
print("Commands: " + ", ".join(f"{key}={relation}" for key, relation in RELATION_SHORTCUTS.items()) + ", s=skip, q=quit")

for i, (_, row) in enumerate(queue_df.iterrows(), start=1):
    print("\n" + "=" * 118)
    print(f"{i}/{len(queue_df)} | {row['pair_id']} | {row['screen_task_id']}: {row['task']}")
    print(f"{'':<44} | {left_judge:<36} | {right_judge:<36}")
    for field in ["relation", "confidence", *DIAGNOSTIC_COLUMNS]:
        print(f"{field:<44} | {str(row[f'{field}__{left_judge}']):<36.36} | {str(row[f'{field}__{right_judge}']):<36.36}")
    print("EXPERT:", row["expert_observed_issue"])
    print("LLM   :", row["llm_observed_issue"])
    bbox = row["expert_bounding_box"]
    show_screen(screen_id=row["screen_id"], bboxes=[bbox] if isinstance(bbox, dict) else [], labels=["expert"], width=300)

    command = input("relation: ").strip().lower()
    if command == "q":
        break
    if command in RELATION_SHORTCUTS:
        manual_relations[row["pair_id"]] = RELATION_SHORTCUTS[command]
        pd.Series(manual_relations, name="manual_relation").rename_axis("pair_id").sort_index().to_csv(LLMJUDGE_MANUAL_OVERRIDES_PATH)
    elif command not in {"", "s"}:
        print("Invalid choice; skipped.")

Manual relations saved so far: 127
Unresolved disagreements: 15,983; queued now: 0
Commands: e=equivalent, a=A_broader, b=B_broader, p=partial_overlap, r=related_but_different, n=no_match, c=contradiction, s=skip, q=quit


## 4.8 Finalize Relations

Judge agreement first, then manual relations; the remaining disagreements take the more confident judge's relation. Confidence ties go to the relation listed first in `RELATION_FALLBACK_RANK` (`contradiction` → `no_match` → `related_but_different` → `partial_overlap` → `equivalent` → `A_broader` → `B_broader`).

Each diagnostic flag is finalized the same way: a flag both judges agree on is kept, otherwise it comes from the judge whose relation became final. It is `disagree` when the judges differ on the flag and both of their relations or neither became final (both: the judges agree on the relation; neither: a manual relation matches no judge). Manually overridden pairs take `same_usability_problem` and `logical_conflict` from the manual relation itself, since those two follow from it.

In [18]:
RELATION_FALLBACK_RANK = {
    relation: rank
    for rank, relation in enumerate([
        "contradiction",
        "no_match",
        "related_but_different",
        "partial_overlap",
        "equivalent",
        "A_broader",
        "B_broader",
    ])
}

llmjudge_final_df = finalize_match_judges(llmjudge_comparison_df, left_judge, right_judge, load_manual_relations())

unresolved = llmjudge_final_df["decision_source"].eq("manual_review")
left_confidence = llmjudge_final_df[f"confidence__{left_judge}"].fillna(-1)
right_confidence = llmjudge_final_df[f"confidence__{right_judge}"].fillna(-1)
left_rank = llmjudge_final_df[left_relation_col].map(RELATION_FALLBACK_RANK).fillna(999)
right_rank = llmjudge_final_df[right_relation_col].map(RELATION_FALLBACK_RANK).fillna(999)
choose_left = (left_confidence > right_confidence) | (left_confidence.eq(right_confidence) & (left_rank <= right_rank))

llmjudge_final_df.loc[unresolved, "relation"] = llmjudge_final_df[left_relation_col].where(choose_left, llmjudge_final_df[right_relation_col])
llmjudge_final_df.loc[unresolved, "confidence"] = left_confidence.where(choose_left, right_confidence)
llmjudge_final_df.loc[unresolved, "decision_source"] = "auto_higher_confidence_fallback"
llmjudge_final_df = finalize_diagnostics(llmjudge_final_df, left_judge, right_judge)

if llmjudge_final_df["relation"].isna().any():
    raise ValueError("Some pairs have no final relation.")
print(llmjudge_final_df["decision_source"].value_counts().sort_index().to_string())
display(llmjudge_final_df["relation"].value_counts().rename_axis("relation").reset_index(name="pairs"))
display(pd.DataFrame({col: llmjudge_final_df[col].value_counts() for col in DIAGNOSTIC_COLUMNS}).fillna(0).astype(int))

decision_source
auto_higher_confidence_fallback    15983
judge_agreement                    48156
manual_override                      127


,relation,pairs
0,no_match,47395
1,related_but_different,13977
2,B_broader,1299
3,equivalent,885
4,A_broader,352
5,partial_overlap,312
6,contradiction,46


,same_target_element,same_usability_problem,comment_a_depends_on_unseen_interaction,comment_b_depends_on_unseen_interaction,logical_conflict
disagree,8065,2197,2726,9924,57
no,34153,58349,60722,51526,64144
partial,16871,2372,482,2526,20
yes,5177,1348,336,290,45


## 4.9 Save LLMJudge Matching Outputs

```text
llmjudge_matching_pairs-{run_label}-{llm_name}.parquet
```

One row per expert-LLM pair with its final relation, confidence and decision source, the final diagnostic flags, and each judge's own relation and flags as `<field>__<judge>` columns.

In [ ]:
matching_pairs_df = pairs_df[[
    "pair_id",
    "screen_task_id",
    "expert_comment_id",
    "expert_observed_issue",
    "llm_comment_id",
    "llm_observed_issue",
]].merge(
    llmjudge_final_df[[
        "pair_id",
        "relation",
        "confidence",
        "decision_source",
        *DIAGNOSTIC_COLUMNS,
        *[f"{field}__{judge}" for judge in (left_judge, right_judge) for field in ["relation", *DIAGNOSTIC_COLUMNS]],
    ]],
    on="pair_id",
    validate="one_to_one",
)
matching_pairs_df.to_parquet(LLMJUDGE_MATCHING_PAIRS_PATH, index=False)

print(f"Saved {len(matching_pairs_df):,} matching pairs to: {LLMJUDGE_MATCHING_PAIRS_PATH}")